# GameTheory-04l : Équilibres de Nash — énumération locale vers nashpy

**Le solveur réduit confronté à l'organe de la littérature, sur des jeux dont on connaît l'issue.**

**Navigation** : [<< 04k-Equilibre-Correle](GameTheory-04k-Equilibre-Correle-Python.ipynb) | [Index](README.md) | [05-ZeroSum-Minimax >>](GameTheory-05-ZeroSum-Minimax-Python.ipynb)

Le carnet [GameTheory-17](GameTheory-17-MultiAgent-RL-Python.ipynb) résout des jeux matriciels **réduits** par le self-play : l'équilibre y est un *point de convergence d'apprentissage*, pas un objet calculé. Le carnet [ChoixSocial-08](SocialChoice/ChoixSocial-08-Kemeny-RankedPairs-Dodgson-Python.ipynb) utilise un solveur (Z3) pour l'agrégation. La littérature théorique des jeux, elle, dispose d'un **organe dédié** : [nashpy](https://nashpy.readthedocs.io/), qui implémente les algorithmes canoniques (énumération de supports, Lemke-Howson).

Ce carnet fait le pont : un **solveur local minimal** (énumération des supports en numpy pur) confronté à **nashpy** sur quatre jeux classiques dont le nombre et la nature des équilibres de Nash sont **connus à l'avance**. Le message : le critère de Nash (`best response` mutuelle) tient en quelques lignes ; l'organe réel donne les **mêmes réponses** sur les jeux où l'issue est connue, et se généralise là où l'énumération locale plafonne.
**Ce que ce carnet ajoute à [GameTheory-04](GameTheory-04-NashEquilibrium-Python.ipynb)** — le carnet canonique des équilibres de Nash utilise **déjà** nashpy (`support_enumeration`, `lemke_howson`), y compris sur pile ou face : ce carnet ne rejoue pas ce geste, il en prend la contrepartie — le même algorithme écrit de zéro, puis confronté à l'organe sur les mêmes jeux. Il pose en outre deux témoins que 04 ne pose pas : le **faux positif** (sans le critère de *best response*, l'énumérateur valide 5 pseudo-équilibres) et la **frontière de complétude** de Lemke-Howson (2 équilibres sur 3 là où il y en a 3).


## 1. Le jeu commun et la définition d'équilibre

Pour qu'une comparaison de solveurs soit **falsifiable**, il faut des jeux dont on connaît **l'issue** — combien d'équilibres, et lesquels. Un équilibre de Nash (NE) d'un jeu à deux joueurs est une paire de stratégies mixtes $(\sigma_1, \sigma_2)$ où **aucun joueur ne peut améliorer son gain espéré en déviant unilatéralement** : chaque $\sigma_i$ est une *best response* à $\sigma_j$. C'est la définition que nous allons **implémenter nous-mêmes** (critère de best response) et que nous allons **comparer** aux algorithmes d'nashpy.

Quatre jeux, tous 2×2, couvrant les quatre **formes** d'équilibre que le critère de Nash peut produire :

| Jeu | Nature | NE attendus (connus) |
|---|---|---|
| **Dilemme du prisonnier** | un NE pur strict | 1 : `(Défaut, Défaut)` |
| **Pile ou face** (jeu à somme nulle) | aucun NE pur | 1 : mixte `(½, ½)` |
| **Chasse au cerf** (Stag Hunt) | deux NE purs + un mixte instable | 3 |
| **Bataille des sexes** | deux NE purs + un mixte | 3 |

In [1]:
import numpy as np
from itertools import combinations

# Quatre jeux 2x2 a gains generaux. A = gains du joueur 1 (lignes), B = gains du joueur 2.
# "attendus" = nombre d'equilibres de Nash connus de la litterature (temoin de falsifiabilite).
JEUX = {
    "Dilemme du prisonnier": dict(
        A=np.array([[3.0, 0.0], [5.0, 1.0]]),
        B=np.array([[3.0, 5.0], [0.0, 1.0]]),
        actions=["Cooperer", "Defaut"],
        attendus=1,   # un unique NE pur : (Defaut, Defaut)
    ),
    "Pile ou face": dict(
        A=np.array([[1.0, -1.0], [-1.0, 1.0]]),
        B=np.array([[-1.0, 1.0], [1.0, -1.0]]),
        actions=["Pile", "Face"],
        attendus=1,   # aucun NE pur ; un unique NE mixte (1/2, 1/2)
    ),
    "Chasse au cerf": dict(
        A=np.array([[4.0, 1.0], [2.0, 3.0]]),
        B=np.array([[4.0, 2.0], [1.0, 3.0]]),
        actions=["Chasser le cerf", "Prendre le lievre"],
        attendus=3,   # 2 NE purs + 1 NE mixte (1/2, 1/2)
    ),
    "Bataille des sexes": dict(
        A=np.array([[2.0, 0.0], [0.0, 1.0]]),
        B=np.array([[1.0, 0.0], [0.0, 2.0]]),
        actions=["Football", "Opera"],
        attendus=3,   # 2 NE purs + 1 NE mixte (2/3, 1/3) vs (1/3, 2/3)
    ),
}

for nom, j in JEUX.items():
    print(f"--- {nom} — actions : {j['actions']} — NE attendus : {j['attendus']}")
    print("A (joueur 1) :", j["A"].tolist(), " B (joueur 2) :", j["B"].tolist())

--- Dilemme du prisonnier — actions : ['Cooperer', 'Defaut'] — NE attendus : 1
A (joueur 1) : [[3.0, 0.0], [5.0, 1.0]]  B (joueur 2) : [[3.0, 5.0], [0.0, 1.0]]
--- Pile ou face — actions : ['Pile', 'Face'] — NE attendus : 1
A (joueur 1) : [[1.0, -1.0], [-1.0, 1.0]]  B (joueur 2) : [[-1.0, 1.0], [1.0, -1.0]]
--- Chasse au cerf — actions : ['Chasser le cerf', 'Prendre le lievre'] — NE attendus : 3
A (joueur 1) : [[4.0, 1.0], [2.0, 3.0]]  B (joueur 2) : [[4.0, 2.0], [1.0, 3.0]]
--- Bataille des sexes — actions : ['Football', 'Opera'] — NE attendus : 3
A (joueur 1) : [[2.0, 0.0], [0.0, 1.0]]  B (joueur 2) : [[1.0, 0.0], [0.0, 2.0]]


### Interprétation : pourquoi ces quatre jeux

Les quatre jeux couvrent ensemble **toutes** les formes d'équilibre que le critère de Nash peut produire sur un jeu 2×2 : un NE **pur strict** (dilemme — la déviation coûte toujours), un **SEUL** équilibre entièrement **mixte** (pile ou face — aucune case n'est stable en pur), et deux jeux à **trois** équilibres (chasse au cerf, bataille des sexes — deux purs + un mixte). Un solveur qui renvoie un nombre différent de celui de la table, ou qui rate un équilibre sur l'un des quatre, est **detecté par construction** : c'est le témoin de falsifiabilité du protocole.

## 2. Le solveur réduit : énumération des supports en numpy pur

L'algorithme local est l'**énumération des supports**, la méthode la plus directe pour les jeux à deux joueurs. L'idée : au voisinage d'un équilibre *fully mixed* sur des supports $S_1$ (joueur 1) et $S_2$ (joueur 2) de même taille $k$, les $2k$ probabilités portées + les 2 gains constants sont **déterminés par un système linéaire** de $2k+2$ équations :

- le joueur 1 est **indifférent** entre les actions de $S_1$ → $k$ équations $A_{S_1,S_2}\,q = c_1$ ;
- le joueur 2 est **indifférent** entre les actions de $S_2$ → $k$ équations $B_{S_1,S_2}^\top p = c_2$ ;
- la normalisation $\sum p = \sum q = 1$ → 2 équations.

On résout ce système pour **chaque** paire de supports $(S_1, S_2)$ de taille $k$ égale ; le candidat n'est retenu que s'il est **positif** (probabilités $\ge 0$) **et** qu'aucune déviation vers une action hors support n'est profitable. Ce second critère — la *best response* — est **le cœur de la définition de Nash** : sans lui, l'indifférence seule est insuffisante (la section 5 le montre par l'absurde).

In [2]:
def _solve_support(A, B, S1, S2, tol=1e-12):
    """Resout le systeme d'indifference sur les supports S1 (j1) et S2 (j2).

    Inconnues : x = [p_{S1} (k), q_{S2} (k), c1, c2]  ->  systeme (2k+2) x (2k+2).
      - indifference j1 : A[S1,S2] . q_{S2} = c1     (colonnes k:2k  = bloc q)
      - indifference j2 : B[S1,S2]^T . p_{S1} = c2   (colonnes 0:k   = bloc p)
      - normalisation : sum p_{S1} = 1,  sum q_{S2} = 1
    Systeme singulier (support degeneré) -> None.
    """
    k = len(S1)
    n = 2 * k + 2
    M = np.zeros((n, n))
    r = np.zeros(n)
    M[0:k, k:2*k]      = A[np.ix_(S1, S2)]     # A q = c1
    M[0:k, 2*k]        = -1.0
    M[k:2*k, 0:k]      = B[np.ix_(S1, S2)].T   # B^T p = c2
    M[k:2*k, 2*k + 1]  = -1.0
    M[2*k, 0:k]        = 1.0                    # sum p = 1
    M[2*k + 1, k:2*k]  = 1.0                    # sum q = 1
    r[2*k] = 1.0
    r[2*k + 1] = 1.0
    if abs(np.linalg.det(M)) < tol:
        return None
    x = np.linalg.solve(M, r)
    p = np.zeros(A.shape[0]); p[list(S1)] = x[0:k]
    q = np.zeros(A.shape[1]); q[list(S2)] = x[k:2*k]
    return p, q


def _est_equilibre(A, B, p, q, tol=1e-7):
    """Critere de Nash : support positif + aucune deviation profitable.

    C'est CE critere (et pas l'indifference seule) qui porte la definition.
    """
    if (p.min() < -tol) or (q.min() < -tol):
        return False
    g1 = A @ q          # gain espéré de j1 par action
    g2 = B.T @ p        # gain espéré de j2 par action
    s1 = np.where(p > tol)[0]   # support effectif de p
    s2 = np.where(q > tol)[0]   # support effectif de q
    best1 = g1[s1].max()
    best2 = g2[s2].max()
    return bool((g1 <= best1 + tol).all() and (g2 <= best2 + tol).all())


def _dedupe(eqs, tol=1e-7):
    """Supprime les points redondants (même équilibre rencontré plusieurs fois)."""
    out = []
    for p, q in eqs:
        if all(np.max(np.abs(p - o[0])) + np.max(np.abs(q - o[1])) > tol for o in out):
            out.append((p, q))
    return out


def nash_local(A, B, tol=1e-7):
    """Énumération des supports : renvoie la liste des NE (p, q) du jeu (A, B)."""
    m, n = A.shape
    trouves = []
    for k in range(1, min(m, n) + 1):
        for S1 in combinations(range(m), k):
            for S2 in combinations(range(n), k):
                cand = _solve_support(A, B, S1, S2)
                if cand is None:
                    continue
                p, q = cand
                if _est_equilibre(A, B, p, q, tol):
                    trouves.append((p, q))
    return _dedupe(trouves, tol)


print("Solveur local défini : _solve_support / _est_equilibre / nash_local / _dedupe")

Solveur local défini : _solve_support / _est_equilibre / nash_local / _dedupe


In [3]:
# Enumeration des equilibres de Nash par le moteur local (nash_local) sur chaque jeu,
# avec controle du nombre d'equilibres (ok = OK si conforme, ECART sinon).
RESULTATS_LOCAL = {}
for nom, j in JEUX.items():
    eqs = nash_local(j["A"], j["B"])
    RESULTATS_LOCAL[nom] = eqs
    ok = "OK" if len(eqs) == j["attendus"] else "ECART"
    print(f"--- {nom} : {len(eqs)} equilibre(s) local  | attendus {j['attendus']}  [{ok}]")
    for i, (p, q) in enumerate(eqs):
        print(f"    NE {i+1}:  p = {np.round(p, 4)}   q = {np.round(q, 4)}")

--- Dilemme du prisonnier : 1 equilibre(s) local  | attendus 1  [OK]
    NE 1:  p = [0. 1.]   q = [0. 1.]
--- Pile ou face : 1 equilibre(s) local  | attendus 1  [OK]
    NE 1:  p = [0.5 0.5]   q = [0.5 0.5]
--- Chasse au cerf : 3 equilibre(s) local  | attendus 3  [OK]
    NE 1:  p = [1. 0.]   q = [1. 0.]
    NE 2:  p = [0. 1.]   q = [0. 1.]
    NE 3:  p = [0.5 0.5]   q = [0.5 0.5]
--- Bataille des sexes : 3 equilibre(s) local  | attendus 3  [OK]
    NE 1:  p = [1. 0.]   q = [1. 0.]
    NE 2:  p = [0. 1.]   q = [0. 1.]
    NE 3:  p = [0.6667 0.3333]   q = [0.3333 0.6667]


### Interprétation : le solveur local trouve les nombres attendus

Sur les quatre jeux, l'énumération locale (numpy pur) renvoie **exactement** les nombres d'équilibres de la table de référence : 1 (dilemme), 1 (pile ou face), 3, 3. On peut déjà lire les valeurs : le dilemme donne `p = q = [0, 1]` (Défaut), pile ou face donne `[0.5, 0.5]` des deux côtés, la chasse au cerf retrouve les deux coins purs **et** le point mixte `[0.5, 0.5]`, et la bataille des sexes `[2/3, 1/3]` vs `[1/3, 2/3]`. Le solveur réduit est donc **suffisant** pour les jeux 2×2 — ce n'est pas une approximation de l'organe, c'est l'organe en germe.

## 3. L'organe réel : nashpy

[nashpy](https://nashpy.readthedocs.io/) est la bibliothèque de référence pour le calcul d'équilibres de Nash — c'est **l'organe** que la littérature théorique appelle. Elle implémente les deux algorithmes canoniques :

- **`Game.support_enumeration()`** — exactement la méthode que l'on vient d'écrire à la main : énumération des supports. C'est l'organe *comparable au millimètre* à notre solveur local ;
- **`Game.lemke_howson()`** — l'algorithme **Lemke-Howson**, une méthode *de suivi de chemin* : au lieu d'énumérer, elle part d'un sommet dégénéré et suit l'arête du polytope jusqu'à un équilibre. C'est un **troisième** algorithme, structurellement différent, qui sert de contrôle indépendant.

On construit le même `nps.Game(A, B)` à partir des **mêmes matrices** `JEUX` — aucun jeu n'est réécrit, l'organe reçoit exactement l'entrée du solveur local.

In [4]:
# Meme enumeration via nashpy (support_enumeration), en contrepoint du moteur local.
import nashpy as nps

NASHPY_SE = {}
for nom, j in JEUX.items():
    g = nps.Game(j["A"], j["B"])
    eqs = list(g.support_enumeration())
    NASHPY_SE[nom] = eqs
    ok = "OK" if len(eqs) == j["attendus"] else "ECART"
    print(f"--- {nom} : {len(eqs)} equilibre(s)  [support_enumeration]  | attendus {j['attendus']}  [{ok}]")
    for i, (p, q) in enumerate(eqs):
        print(f"    NE {i+1}:  p = {np.round(p, 4)}   q = {np.round(q, 4)}")

--- Dilemme du prisonnier : 1 equilibre(s)  [support_enumeration]  | attendus 1  [OK]
    NE 1:  p = [0. 1.]   q = [0. 1.]
--- Pile ou face : 1 equilibre(s)  [support_enumeration]  | attendus 1  [OK]
    NE 1:  p = [0.5 0.5]   q = [0.5 0.5]
--- Chasse au cerf : 3 equilibre(s)  [support_enumeration]  | attendus 3  [OK]
    NE 1:  p = [1. 0.]   q = [1. 0.]
    NE 2:  p = [0. 1.]   q = [0. 1.]
    NE 3:  p = [0.5 0.5]   q = [0.5 0.5]
--- Bataille des sexes : 3 equilibre(s)  [support_enumeration]  | attendus 3  [OK]
    NE 1:  p = [1. 0.]   q = [1. 0.]
    NE 2:  p = [0. 1.]   q = [0. 1.]
    NE 3:  p = [0.6667 0.3333]   q = [0.3333 0.6667]


In [5]:
# Lemke-Howson : un point de depart par label "initially dropped" (ici 0..3 sur un 2x2).
# C'est une methode de SUIVI DE CHEMIN — elle garantit d'atteindre UN equilibre,
# pas necessairement TOUS. On deduplique les resultats des differents departs.
NASHPY_LH = {}
for nom, j in JEUX.items():
    g = nps.Game(j["A"], j["B"])
    bruts = []
    for label in range(j["A"].shape[0] + j["B"].shape[0]):
        try:
            bruts.append(tuple(g.lemke_howson(initial_dropped_label=label)))
        except Exception as exc:
            print(f"    [warn] {nom} label {label} : {exc}")
    dedup = _dedupe([tuple(t) for t in bruts])
    NASHPY_LH[nom] = dedup
    print(f"--- {nom} : {len(dedup)} equilibre(s) distinct(s)  [lemke_howson, 4 depart]  | attendus {j['attendus']}")
    for i, (p, q) in enumerate(dedup):
        print(f"    NE {i+1}:  p = {np.round(p, 4)}   q = {np.round(q, 4)}")

--- Dilemme du prisonnier : 1 equilibre(s) distinct(s)  [lemke_howson, 4 depart]  | attendus 1
    NE 1:  p = [0. 1.]   q = [0. 1.]
--- Pile ou face : 1 equilibre(s) distinct(s)  [lemke_howson, 4 depart]  | attendus 1
    NE 1:  p = [0.5 0.5]   q = [0.5 0.5]
--- Chasse au cerf : 2 equilibre(s) distinct(s)  [lemke_howson, 4 depart]  | attendus 3
    NE 1:  p = [1. 0.]   q = [1. 0.]
    NE 2:  p = [0. 1.]   q = [0. 1.]
--- Bataille des sexes : 2 equilibre(s) distinct(s)  [lemke_howson, 4 depart]  | attendus 3
    NE 1:  p = [1. 0.]   q = [1. 0.]
    NE 2:  p = [0. 1.]   q = [0. 1.]


### Interprétation : l'organe retrouve les mêmes nombres

Les deux algorithmes d'nashpy reproduisent les nombres attendus. **`support_enumeration`** est le jumeau direct du solveur local — on les comparera point par point à la section 4. **`lemke_howson`** est plus prudent : en tant que suivi de chemin, un *seul* départ peut rater un équilibre sur les jeux à plusieurs NE. Ici les 4 départs distincts couvrent les jeux à 3 équilibres (chasse, bataille) ; sur un jeu plus dégénéré il faudrait plus de départs. C'est exactement la **frontière de généralisation** que ce carnet veut rendre visible : l'organe réel se généralise là où l'énumération locale plafonne.

## 4. La confrontation : trois algorithmes, une issue connue

Le cœur de la validation. On met **en regard** : (i) le solveur **local**, (ii) **nashpy `support_enumeration`**, (iii) **nashpy `lemke_howson`** — sur les quatre jeux dont l'issue est connue.

Deux **niveaux de rigueur**, distincts :

- **Parité d'ensembles** (i) ↔ (ii) : les deux *énumérateurs complets* doivent renvoyer **exactement les mêmes points** (même cardinal, même ensemble à la tolérance près). C'est la comparaison au millimètre du solveur réduit avec son jumeau dans l'organe.
- **Saine sortie** pour (iii) : `lemke_howson` ne garantissant pas l'énumération *complète*, on ne l'exige pas égale — on exige seulement que **chaque** point qu'il renvoie **soit bien un équilibre** (vérifié par notre critère de best response). Un suivi de chemin peut en manquer, il n'en invente pas.

Le **témoin de cardinal** (colonne `attendus`) ferme la boucle contre la littérature : si les trois s'accordaient mais se trompaient ensemble, le compte attendu le révélerait.

In [6]:
def _meme_point(p1, q1, p2, q2, tol=1e-5):
    """Deux equilibres coïncident-ils (à la tolérance près) ?"""
    return bool(np.max(np.abs(p1 - p2)) <= tol and np.max(np.abs(q1 - q2)) <= tol)

def _dans_ensemble(p, q, eqs, tol=1e-5):
    """Le point (p, q) est-il déjà dans la liste eqs ?"""
    return any(_meme_point(p, q, u[0], u[1], tol) for u in eqs)

def _ensembles_egaux(e1, e2, tol=1e-5):
    """Même cardinal + chaque point de l'un est dans l'autre."""
    if len(e1) != len(e2):
        return False
    return all(_dans_ensemble(p, q, e2, tol) for p, q in e1)

print(f"{'jeu':<22}{'attendus':>9}{'local':>7}{'nashpy-SE':>11}{'nashpy-LH':>11}   {'parité i<->ii':>13}   saineté iii")
print("-" * 78)
tous_ok = True
for nom, j in JEUX.items():
    e_loc = RESULTATS_LOCAL[nom]
    e_se  = NASHPY_SE[nom]
    e_lh  = NASHPY_LH[nom]
    parity = _ensembles_egaux(e_loc, e_se)          # (i) == (ii) : énumérateurs complets
    sain   = len(e_lh) > 0 and all(_est_equilibre(j["A"], j["B"], p, q) for p, q in e_lh)  # (iii) saine
    compte = len(e_loc) == j["attendus"]
    tous_ok = tous_ok and parity and sain and compte
    print(f"{nom:<22}{j['attendus']:>9}{len(e_loc):>7}{len(e_se):>11}{len(e_lh):>11}"
          f"   {'OK' if parity else 'ECART':>13}   {'OK' if sain else 'SALISSAGE':>9}")

print("-" * 78)
print("Verdict :", "PARITE — les 3 familles tombent sur les mêmes équilibres" if tous_ok
      else "ECART — protocole déçu (voir colonnes)")

jeu                    attendus  local  nashpy-SE  nashpy-LH   parité i<->ii   saineté iii
------------------------------------------------------------------------------
Dilemme du prisonnier         1      1          1          1              OK          OK
Pile ou face                  1      1          1          1              OK          OK
Chasse au cerf                3      3          3          2              OK          OK
Bataille des sexes            3      3          3          2              OK          OK
------------------------------------------------------------------------------
Verdict : PARITE — les 3 familles tombent sur les mêmes équilibres


### Interprétation : le témoin positif — parité mesurée

`PARITE` sur les quatre jeux. La ligne de lecture compte double :

1. **Le solveur local et l'organe `support_enumeration` coïncident point par point** — même cardinal, même ensemble. Ce n'est pas « des nombres qui s'accordent », c'est **les mêmes points** : le code minimal et la bibliothèque de référence produisent le même objet. Le critère de Nash tient bien en quelques lignes de numpy.
2. **`lemke_howson` ne salit jamais** : chaque point qu'il renvoie est vérifié équilibre par *notre* critère indépendant. Un algorithme structurellement différent (suivi de chemin vs énumération) atterrit sur les mêmes solutions — un contrôle croisé.

C'est le **témoin positif** : la mesure a ce qu'elle doit avoir *là où elle doit l'avoir*.

## 5. Le témoin négatif : retirer le critère de Nash

Un protocole n'est **falsifiable** que s'il peut être **déçu**. On le prouve en **cassant** le solveur : on garde l'énumération des supports et la résolution du système d'indifférence, mais on **supprime le critère de best response** (`_est_equilibre`). Sans lui, *tout* candidat d'indifférence est accepté — y compris ceux qui ne sont **pas** des équilibres.

On l'applique à **pile ou face**, le jeu sans équilibre pur : la version cassée va alors « trouver » des équilibres purs imaginaires. L'écart entre ce qu'elle rend et la vérité (1 équilibre mixte) **est** la valeur du critère de best response qu'on vient de retirer.

In [7]:
def nash_local_CASSE(A, B, tol=1e-7):
    """Version CASSEE : même énumération, même résolution d'indifférence,
    mais SANS le critère de best response. Tout candidat est accepté."""
    m, n = A.shape
    trouves = []
    for k in range(1, min(m, n) + 1):
        for S1 in combinations(range(m), k):
            for S2 in combinations(range(n), k):
                cand = _solve_support(A, B, S1, S2)
                if cand is None:
                    continue
                trouves.append(cand)      # <-- _est_equilibre(...) volontairement absent
    return _dedupe(trouves, tol)

j = JEUX["Pile ou face"]
casses  = nash_local_CASSE(j["A"], j["B"])
vrais   = RESULTATS_LOCAL["Pile ou face"]

print(f"Pile ou face — énumérateur SANS le critère de Nash : {len(casses)} 'équilibres' rendus")
for i, (p, q) in enumerate(casses):
    est_ne = _est_equilibre(j["A"], j["B"], p, q)
    print(f"    cand {i+1}:  p = {np.round(p, 4)}  q = {np.round(q, 4)}   NE réel ? {est_ne}")
print()
print(f"Avec le critère de best response (solveur complet) : {len(vrais)} équilibre(s) — la vérité")

Pile ou face — énumérateur SANS le critère de Nash : 5 'équilibres' rendus
    cand 1:  p = [1. 0.]  q = [1. 0.]   NE réel ? False
    cand 2:  p = [1. 0.]  q = [0. 1.]   NE réel ? False
    cand 3:  p = [0. 1.]  q = [1. 0.]   NE réel ? False
    cand 4:  p = [0. 1.]  q = [0. 1.]   NE réel ? False
    cand 5:  p = [0.5 0.5]  q = [0.5 0.5]   NE réel ? True

Avec le critère de best response (solveur complet) : 1 équilibre(s) — la vérité


### Interprétation : le témoin négatif — la mesure peut se tromper

La version cassée rend **cinq** « équilibres » sur pile ou face : les quatre profils purs `(Pile,Pile)`, `(Pile,Face)`, `(Face,Pile)`, `(Face,Face)` **et** le mixte `[0.5, 0.5]`. Un seul est un vrai équilibre — le mixte. Les quatre purs sont de **pures indifférences** (chaque joueur est indifférent entre ses deux actions face à la stratégie de l'autre), mais **aucun n'est stable** : un déviateur y gagne. C'est exactement le trou que le critère de best response referme.

Le message : **l'indifférence n'est pas la stabilité.** Retirer une seule ligne du solveur le fait « raisonner » sur 5 points au lieu d'un. C'est ce qui rend la parité de la section 4 significative — elle montre que l'on a implémenté *toute* la définition, pas seulement l'algèbre linéaire.

## 6. Exercices

In [8]:
# Exercice 1 — le coût combinatoire de l'énumération
# L'énumération des supports teste chaque paire (S1, S2). Compte combien de
# systèmes linéaires sont résolvés sur un jeu 2x2 vs un jeu 3x3, et montre
# pourquoi cette méthode "plafonne" quand la taille des actions croît
# (le coût croît en binom(m,k)*binom(n,k) — explosif).
# TODO etudiant
def exercice_cout_enumeration(A, B):
    # Indice : compte les itérations de la boucle triple de nash_local.
    print("Exercice a completer")
    return None

In [9]:
# Exercice 2 — la valeur d'un jeu à somme nulle
# Pile ou face est à somme nulle : A = -B. nashpy propose Game.minmax() /
# Game.maxmin() qui renvoient la "valeur" du jeu et une stratégie mixte
# optimale. Montre que la valeur du pile-ou-face est 0 (parité parfaite)
# et que les stratégies mixtes optimales sont (1/2, 1/2).
# TODO etudiant
def exercice_valeur_zero_sum():
    # Indice : nps.Game(A, B) puis .minmax() — la valeur attendue est 0.
    print("Exercice a completer")
    return None

In [10]:
# Exercice 3 — dynamique de best response sur un jeu à plusieurs NE
# Sur la chasse au cerf (2 NE purs), implémente l'itération de best
# response : chaque joueur joue une réponse optimales à l'action de l'autre.
# Montre que la convergence vers UN des deux NE purs dépend de l'état
# initial (les NE sont des "bassins d'attraction" distincts) — un lien
# direct avec le self-play du carnet GameTheory-17.
# TODO etudiant
def exercice_best_response_dynamique(A, B, init=(0, 0), n_iter=20):
    # Indice : best response de j1 = argmax de A @ q ; itère jusqu'à fixe point.
    print("Exercice a completer")
    return None

## 7. Résumé et points clés

- **Le critère de Nash tient en quelques lignes.** L'énumération des supports tient en quelques lignes de numpy : un système linéaire par paire de supports + un test de best response. Sur les jeux 2×2, ce solveur minimal est **suffisant**.
- **L'organe réel donne les mêmes réponses.** `nashpy.support_enumeration` coïncide **point par point** avec le solveur local sur les quatre jeux ; `lemke_howson` (un 3ᵉ algorithme, de suivi de chemin) n'en **invente aucun** — un contrôle croisé. La parité est *mesurée*, pas supposée.
- **Le protocole est falsifiable.** Témoin de cardinal (les nombres connus de la littérature) + témoin **négatif** (retirer le critère de best response → l'« énumérateur » salit avec des pseudo-équilibres). L'indifférence n'est pas la stabilité.
- **La frontière de généralisation.** L'énumération locale plafonne (coût combinatoire) ; l'organe se généralise — `lemke_howson` suit un chemin au lieu d'énumérer. C'est le même passage d'échelle que GT-24 a montré entre modèle réduit et framework : **l'interface change, l'objet calculé reste le même.**

**Où ça mène** : les jeux à 3+ joueurs, où l'énumération des supports devient exponentielle et où d'autres organes (`nashpy` sur des jeux plus grands, ou les solveurs de points intérieurs) prennent le relais — la série GameTheory continue.

## Références

- [nashpy — documentation](https://nashpy.readthedocs.io/) : l'organe de calcul d'équilibres de Nash (support enumeration, Lemke-Howson).
- [GameTheory-17 — Apprentissage par Renforcement Multi-Agent](GameTheory-17-MultiAgent-RL-Python.ipynb) : le self-play réduit (Fictitious Play, NFSP, PSRO) dont le présent carnet fournit la *caractérisation* d'équilibre.
- [GameTheory-24 — Self-Play Tabulaire vers PettingZoo](GameTheory-24-SelfPlay-Tabulaire-vers-PettingZoo-Python.ipynb) : le pont entre modèle réduit et organe framework, sur l'apprentissage.
- Osborne & Rubinstein, *A Course in Game Theory* — chapitres sur l'énumération des supports et l'algorithme de Lemke-Howson.